# 05 Reporting and metrics
Run a short mock simulation, build a `ReportBuilder`, and plot turn counts.

In [ ]:
from pathlib import Path
import sys
for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src = base / "src"
    if (src / "agent_rpg").is_dir():
        if str(src) not in sys.path:
            sys.path.insert(0, str(src))
        break
from agent_rpg.repo_root import find_repo_root
ROOT = find_repo_root()
print("Repository root:", ROOT)


In [ ]:
from agent_rpg import ReportBuilder, SimulationEngine, load_scenario
from agent_rpg.backends.fake import FakeLLMBackend

scenario = load_scenario(ROOT / "examples" / "scenarios" / "minimal.yaml")
scenario.orchestration.enable_thought_phase = False
scenario.orchestration.max_rounds = 1
scenario.world.max_rounds = 1
out = SimulationEngine(scenario).run(
    FakeLLMBackend(
        factory=lambda i, m: '{\"thought\":\"\",\"say\":\"hello there\",\"directed_at\":\"bob\"}',
    ),
    output_dir=ROOT / "runs",
    run_id="nb05_metrics",
)
rb = ReportBuilder.from_jsonl(out / "events.jsonl")
d = rb.to_dict(scenario=scenario)
print(d["summary"])
print(d["social_dynamics"]["turn_counts"])

In [ ]:
try:
    import matplotlib.pyplot as plt

    counts = d["social_dynamics"]["turn_counts"]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(list(counts.keys()), list(counts.values()))
    ax.set_title("Messages per agent")
    plt.show()
except Exception as e:
    print("Plot skipped:", e)